# 05 — Select the validation calling policy

The cancellation model is fixed as logistic regression with `C = 1`. This notebook decides how its booking-time risk score should be turned into the weekly call list. It compares three policies on validation and selects one for final evaluation.

No probability calibration is added. The raw `p_cancel` is treated as a risk score, and any score that combines risk, value, and urgency is a priority index—not an exact cancellation probability or expected-dollar estimate. Holdout is not selected, scored, or evaluated.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hotel_bookings.csv").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "modeling"))

import evaluation_policy as evaluation
from models import make_logistic_pipeline
from prepare_data import (
    RAW_DATA_PATH, complete_180_rows, model_features, prepare_data,
)

pd.set_option("display.max_columns", 60)

## 1. Recreate the fixed model and validation population

The model is fitted on the same 43,264 training rows whose outcomes were known by validation start. It produces one score for each of the same 11,050 validation bookings. The score is calculated from approved booking-time features and remains unchanged while a booking waits in the queue.

Logistic regression remains the selected model because CatBoost reached only five additional cancellations in 1,950 validation calls, while logistic regression is easier to explain.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
prepared = prepare_data(raw)
modeling_population = complete_180_rows(prepared)

validation_population = modeling_population.loc[
    modeling_population["booking_period_split"].eq("validation")
].copy()
validation_start = validation_population["booking_week_start"].min()

nominal_train = modeling_population.loc[
    modeling_population["booking_period_split"].eq("train")
].copy()
model_fit_population = nominal_train.loc[
    nominal_train["arrival_date"].lt(validation_start)
].copy()

X_fit = model_features(model_fit_population)
y_fit = model_fit_population["is_canceled"].copy()
X_validation = model_features(validation_population)

logistic_model = make_logistic_pipeline(c_value=1.0)
logistic_model.fit(X_fit, y_fit)

validation_scored = validation_population.copy()
validation_scored["p_cancel"] = logistic_model.predict_proba(
    X_validation
)[:, 1]

display(
    pd.Series(
        {
            "model fitting rows": len(model_fit_population),
            "validation rows": len(validation_population),
            "validation booking weeks": validation_population[
                "booking_week_start"
            ].nunique(),
        },
        name="count",
    ).to_frame()
)
display(
    validation_scored["p_cancel"]
    .agg(["min", "mean", "median", "max"])
    .rename("fixed booking-time risk score")
    .to_frame()
)

,count
model fitting rows,43264
validation rows,11050
validation booking weeks,13


,fixed booking-time risk score
min,4.178394e-59
mean,2.119707e-01
median,9.357881e-02
max,9.994424e-01


## 2. Compare three queue priorities

Each policy starts its own fresh validation simulation and follows the same Monday eligibility, 150-call capacity, carryover, and one-call-only rules. Only the ranking score changes:

1. **Risk only:** `p_cancel`
2. **Risk x booking value:** `p_cancel x max(adr, 0) x total_nights`
3. **Risk x value x urgency:** the value score multiplied by a mild 30-day urgency boost from 1.00 to 1.50

All three rank on booking-time information only. Precision@150 counts how many cancellations a policy reaches; realized loss reached says how expensive those cancellations were. ADR times nights is booked room-revenue exposure, not profit.

Realized loss is measured from the actual cancellation date under the 30-day resale assumption fixed in notebook 02, with the no-resale bound reported beside it. Because that valuation depends on when the guest canceled rather than on when we called, it is not mechanically inflated by the urgency term in policy 3. That circularity was the flaw in the earlier version of this comparison: the only metric favouring the urgency boost was one built from the same proximity term the boost ranks on.

In [3]:
policy_labels = {
    "risk_only": "risk only",
    "risk_and_value": "risk x booking value",
    "risk_value_and_urgency": "risk x value x mild 30-day urgency",
}


def summarize_policy(label, calls, aggregate):
    return {
        "policy": label,
        "calls": len(calls),
        "cancellations reached": int(aggregate["cancellations reached"]),
        "mean weekly precision@150": aggregate[
            "mean weekly queue precision at capacity"
        ],
        "realized loss reached": aggregate["realized loss reached"],
        "full value loss reached": aggregate["full value loss reached"],
        "average days to arrival": calls[
            "days_to_arrival_at_call"
        ].mean(),
        "capacity utilization": aggregate[
            "mean capacity utilization"
        ],
        "bookings waiting after final decision": aggregate[
            "bookings still waiting after final decision"
        ],
    }


policy_results = {}
policy_summary_rows = []

for policy, label in policy_labels.items():
    calls, weekly, aggregate, final_queue = (
        evaluation.simulate_weekly_policy(
            validation_scored,
            "validation",
            policy=policy,
            urgency_window_days=30,
        )
    )
    policy_results[policy] = {
        "calls": calls,
        "weekly": weekly,
        "aggregate": aggregate,
        "final_queue": final_queue,
    }
    policy_summary_rows.append(
        summarize_policy(label, calls, aggregate)
    )

policy_summary = pd.DataFrame(policy_summary_rows).set_index("policy")
policy_summary.style.format(
    {
        "mean weekly precision@150": "{:.2%}",
        "realized loss reached": "{:,.0f}",
        "full value loss reached": "{:,.0f}",
        "average days to arrival": "{:.1f}",
        "capacity utilization": "{:.2%}",
        "bookings waiting after final decision": "{:,.0f}",
    }
)

,calls,cancellations reached,mean weekly precision@150,realized loss reached,full value loss reached,average days to arrival,capacity utilization,bookings waiting after final decision
policy,,,,,,,,
risk only,1950,1132,58.05%,"135,542","437,507",86.0,100.00%,"2,583"
risk x booking value,1950,1055,54.10%,"147,329","492,986",85.2,100.00%,"2,609"
risk x value x mild 30-day urgency,1950,1046,53.64%,"150,925","493,069",81.7,100.00%,"2,649"


### Weekly precision and the random reference

The weekly table shows whether a policy's average is driven by only a few weeks. The repeated-random result provides the same capacity-matched reference used in the model notebooks.

In [4]:
weekly_precision = pd.concat(
    [
        result["weekly"][["call_date", "precision_at_capacity"]]
        .assign(policy=policy_labels[policy])
        for policy, result in policy_results.items()
    ],
    ignore_index=True,
).pivot(
    index="call_date", columns="policy", values="precision_at_capacity"
)
display(weekly_precision.style.format("{:.2%}"))

random_results = evaluation.repeated_random_baseline(
    validation_scored, "validation", repeats=100
)
random_metric = "mean weekly queue precision at capacity"
display(
    pd.Series(
        {
            "random mean": random_results[random_metric].mean(),
            "random 5th percentile": random_results[
                random_metric
            ].quantile(0.05),
            "random 95th percentile": random_results[
                random_metric
            ].quantile(0.95),
        },
        name="mean weekly precision@150",
    ).map(lambda value: f"{value:.2%}").to_frame()
)

policy,risk only,risk x booking value,risk x value x mild 30-day urgency
call_date,,,
2016-09-05 00:00:00,56.00%,52.67%,50.67%
2016-09-12 00:00:00,47.33%,38.67%,40.00%
2016-09-19 00:00:00,63.33%,56.00%,56.67%
2016-09-26 00:00:00,52.67%,48.00%,47.33%
2016-10-03 00:00:00,42.67%,39.33%,37.33%
2016-10-10 00:00:00,52.67%,45.33%,45.33%
2016-10-17 00:00:00,52.00%,52.00%,50.67%
2016-10-24 00:00:00,66.00%,67.33%,68.67%
2016-10-31 00:00:00,61.33%,43.33%,40.67%


,mean weekly precision@150
random mean,24.84%
random 5th percentile,23.47%
random 95th percentile,25.90%


### Example of a value-weighted Monday call list

The first ten rows of the risk x value queue show why a booking reaches the top: a high fixed cancellation risk, a large booked amount, or both. `days_to_arrival_at_call` is shown for context only; this ranking does not use it. `source_row_id` is only a source-row key and would need to be replaced by a real reservation ID in practice.

In [5]:
proposed_calls = policy_results["risk_and_value"]["calls"]
first_call_date = proposed_calls["call_date"].min()
example_call_list = proposed_calls.loc[
    proposed_calls["call_date"].eq(first_call_date),
    [
        "call_rank",
        "source_row_id",
        "p_cancel",
        "booking_value_proxy",
        "days_to_arrival_at_call",
        "risk_value_priority",
    ],
].head(10)
example_call_list.style.format(
    {
        "p_cancel": "{:.2%}",
        "booking_value_proxy": "{:,.0f}",
        "risk_value_priority": "{:,.1f}",
    }
)

,call_rank,source_row_id,p_cancel,booking_value_proxy,days_to_arrival_at_call,risk_value_priority
0,1,57099,79.94%,"3,306",13,"2,642.9"
1,2,34275,87.15%,"1,113",173,970.0
2,3,32753,45.48%,"1,522",125,692.2
3,4,62792,44.81%,"1,190",139,533.2
4,5,57595,35.90%,"1,396",23,501.2
5,6,60052,32.93%,"1,357",60,446.9
6,7,98447,20.90%,"1,980",7,413.7
7,8,59241,56.80%,726,48,412.5
8,9,60147,53.46%,751,62,401.5
9,10,8850,34.60%,"1,158",41,400.7


## 3. Keep the urgency-window sensitivity predeclared

The urgency arm uses the mild 30-day boost declared in notebook 02. The same queue is also run with 14- and 60-day boost windows to show how the proximity definition changes the calls. These alternatives are not searched for a validation winner.

Every column below is now comparable across the three rows. The realized-loss valuation uses the fixed 30-day resale assumption and the actual cancellation date, so it no longer changes definition when the ranking's urgency window changes. In the earlier version it did, which made these rows incomparable and hid how little the window matters.

In [6]:
urgency_rows = []
urgency_results = {}

for window_days in [14, 30, 60]:
    calls, weekly, aggregate, final_queue = (
        evaluation.simulate_weekly_policy(
            validation_scored,
            "validation",
            policy="risk_value_and_urgency",
            urgency_window_days=window_days,
        )
    )
    urgency_results[window_days] = {
        "calls": calls,
        "weekly": weekly,
        "aggregate": aggregate,
        "final_queue": final_queue,
    }
    summary = summarize_policy(
        f"{window_days}-day window", calls, aggregate
    )
    urgency_rows.append(
        {
            "urgency window": summary["policy"],
            "calls": summary["calls"],
            "cancellations reached": summary["cancellations reached"],
            "mean weekly precision@150": summary[
                "mean weekly precision@150"
            ],
            "realized loss reached": summary["realized loss reached"],
            "average days to arrival": summary[
                "average days to arrival"
            ],
            "capacity utilization": summary["capacity utilization"],
        }
    )

urgency_summary = pd.DataFrame(urgency_rows).set_index("urgency window")
urgency_summary.style.format(
    {
        "mean weekly precision@150": "{:.2%}",
        "realized loss reached": "{:,.0f}",
        "average days to arrival": "{:.1f}",
        "capacity utilization": "{:.2%}",
    }
)

,calls,cancellations reached,mean weekly precision@150,realized loss reached,average days to arrival,capacity utilization
urgency window,,,,,,
14-day window,1950,1049,53.79%,"149,424",82.5,100.00%
30-day window,1950,1046,53.64%,"150,925",81.7,100.00%
60-day window,1950,1048,53.74%,"153,094",79.9,100.00%


## 4. Select the policy

The **risk x booking value** policy is selected for final evaluation.

**Against risk-only**, it gives up 3.95 points of precision@150 (54.10% versus 58.05%, or 1,055 versus 1,132 cancellations reached) and buys 8.7% more realized loss reached (147,329 versus 135,542) and 12.7% more canceled booking value (492,986 versus 437,507). That is the trade this program wants. The goal is protecting revenue, not maximizing the count of guests contacted, and a cancellation is not equally expensive whatever its size. Both policies stay far above the 24.84% random reference.

**Against risk x value x urgency**, the mild boost buys almost nothing. It reaches 2.4% more realized loss (150,925 versus 147,329) and an indistinguishable amount of booking value (493,069 versus 492,986, a difference of $83 on nearly half a million), while costing another 0.46 points of precision and nine more cancellations. The observed effect is small: the boost is exactly 1.00 for most of the list, and average days to arrival moves only from 85.2 to 81.7. Bookings are scored at booking time and become eligible on the following Monday; ranking and capacity determine whether they are called then, later, or not during the evaluation period. Under notebook 02's declared rule, materially similar performance favours the simpler ranking, so the urgency term and its window parameter are dropped. This is the same tiebreak that chose logistic regression over CatBoost, applied consistently.

**What this does not settle.** Bookings become eligible on the Monday after their booking week. Each weekly list selects up to 150 active bookings, so a booking can be called on its first eligible Monday, on a later Monday, or remain uncalled. The selected lists still tend to contact bookings months before arrival, leaving little proximity for the urgency boost to detect. A different design that deliberately holds high-risk bookings and calls them near arrival is not one of these three policies and is not tested by this comparison. It would need cancellation timing represented in the ranking objective, which none of these policies do. Notebook 06 records it as a next step rather than an untested claim.

The 14- and 60-day windows remain sensitivity checks and are close to the 30-day result, which is consistent with the boost being largely inert under this queue design. Notebook 06 applies the selected model and policy to holdout once, then runs the explicit return scenarios under both loss bounds.